In [5]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash
from CRUD_Python_Module import AnimalShelter
# Configure the necessary Python module imports
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output
JupyterDash.infer_jupyter_proxy_config()

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# change animal_shelter and AnimalShelter to match your CRUD Python module file name and class name
from CRUD_Python_Module import AnimalShelter

###########################
# Data Manipulation / Model
###########################

username = "aacuser"
password = "Password123"
shelter = AnimalShelter(username, password)

records = shelter.read({})
records = shelter.read({})

# Convert cursor to list if needed
if records is None:
    records = []
elif not isinstance(records, list):
    records = list(records)

# Build DataFrame safely
if len(records) == 0:
    df = pd.DataFrame()
else:
    df = pd.DataFrame.from_records(records)
    if '_id' in df.columns:
        df.drop(columns=['_id'], inplace=True)



#########################
# Dashboard Layout / View
#########################

app = JupyterDash('GraziosoDashboard')

# Load logo
image_filename = "Grazioso Salvare Logo.png"
encoded_image = base64.b64encode(open(image_filename, 'rb').read()).decode()

app.layout = html.Div([
    html.Center(html.B(html.H1('Grazioso Salvare Dashboard'))),
    html.Center(html.Img(src='data:image/png;base64,{}'.format(encoded_image),
                         style={'width':'200px'})),
    html.Center(html.H3("Created by Tevaen Kenerly")),
    html.Hr(),

    # Rescue filter options
    dcc.RadioItems(
        id='filter-type',
        options=[
            {'label': 'Reset', 'value': 'reset'},
            {'label': 'Water Rescue', 'value': 'water'},
            {'label': 'Mountain/Wilderness Rescue', 'value': 'mountain'},
            {'label': 'Disaster/Individual Tracking', 'value': 'disaster'}
        ],
        value='reset',
        inline=True
    ),

    html.Hr(),

    # Data table
    dash_table.DataTable(
        id='datatable-id',
        columns=[{"name": i, "id": i} for i in df.columns],
        data=df.to_dict('records'),
        page_size=10,
        sort_action='native',
        filter_action='native',
        row_selectable='single',
        selected_rows=[0]
    ),

    html.Br(),
    html.Hr(),

    # Charts side-by-side
    html.Div(className='row', style={'display': 'flex'}, children=[
        html.Div(id='graph-id', className='col s12 m6'),
        html.Div(id='map-id', className='col s12 m6')
    ])
])

#############################################
# Interaction Between Components / Controller
#############################################

# Filter callback
@app.callback(Output('datatable-id','data'),
              [Input('filter-type', 'value')])
def update_dashboard(filter_type):

    if filter_type == 'water':
        query = {
            "animal_type":"Dog",
            "breed":{"$in":["Labrador Retriever Mix","Chesapeake Bay Retriever","Newfoundland"]},
            "sex_upon_outcome":"Intact Female",
            "age_upon_outcome_in_weeks":{"$gte":26,"$lte":156}
        }

    elif filter_type == 'mountain':
        query = {
            "animal_type":"Dog",
            "breed":{"$in":["German Shepherd","Alaskan Malamute","Siberian Husky","Rottweiler"]},
            "sex_upon_outcome":"Intact Male",
            "age_upon_outcome_in_weeks":{"$gte":26,"$lte":156}
        }

    elif filter_type == 'disaster':
        query = {
            "animal_type":"Dog",
            "breed":{"$in":["Doberman Pinscher","German Shepherd","Golden Retriever"]},
            "sex_upon_outcome":"Intact Male",
            "age_upon_outcome_in_weeks":{"$gte":20,"$lte":300}
        }

    else:
        query = {}

    dff = pd.DataFrame.from_records(shelter.read(query))
    if '_id' in dff.columns:
        dff.drop(columns=['_id'], inplace=True)

    return dff.to_dict('records')


# Pie chart callback
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")]
)
def update_graphs(viewData):
    if viewData is None:
        dff = df
    else:
        dff = pd.DataFrame.from_dict(viewData)

    fig = px.pie(dff, names='breed', title='Breed Distribution')
    return [dcc.Graph(figure=fig)]


# Highlight selected column
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    return [{
        'if': { 'column_id': i },
        'background_color': '#D2F3FF'
    } for i in selected_columns]


# Map callback
@app.callback(
    Output('map-id', 'children'),
    [Input('datatable-id', 'derived_virtual_data'),
     Input('datatable-id', 'derived_virtual_selected_rows')]
)
def update_map(viewData, index):
    if viewData is None or index is None:
        return

    dff = pd.DataFrame.from_dict(viewData)
    row = index[0]

    return [
        dl.Map(style={'width': '1000px', 'height': '500px'},
               center=[30.75,-97.48], zoom=10, children=[
            dl.TileLayer(),
            dl.Marker(position=[dff.iloc[row,13], dff.iloc[row,14]], children=[
                dl.Tooltip(dff.iloc[row,4]),
                dl.Popup([
                    html.H1("Animal Name"),
                    html.P(dff.iloc[row,9])
                ])
            ])
        ])
    ]

app.run_server()



Dash app running on https://camelstudent-smileleonid-3000.codio.io/proxy/8050/
